<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 04: Batch inference for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook downloads a stored model and uses it to perform batch inference.

It performs the following steps:

1. 


### 📝 Imports

In [5]:
# top of notebook
from features import build_features, add_calendar_features, add_train_lag_features, \
     add_station_network_state_features, detect_trigger_time, \
     add_reactive_early_dynamics, add_weather_rolling_features_if_present


In [6]:
import os
import datetime
import pandas as pd
import requests
import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
import hopsworks
from typing import Any, Dict, List, Optional, Tuple


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

## 📡 Connect to Hopsworks Feature Store

In [7]:
# Optional: Hopsworks storage (not required)
try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))

#train_feature_df = project.get("train_stop_events_labeled")

#uncoment the below line when weather features are stored
#weather_df = project.get("weather_features") 


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-05 17:14:02,387 INFO: Initializing external client
2026-01-05 17:14:02,389 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-05 17:14:03,631 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2182
Hopsworks login OK


In [8]:
import os
import json
import datetime as dt
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd
import joblib


## 📥 Inputs

This notebook runs **batch inference** (no training) using:

- A **canonical events table**: `train_stop_events_labeled.parquet` (from Part 01), filtered to an inference window  
  *(or you can point to any new ops/events parquet with the same schema).*
- The **same feature engineering** as Part 02 (copied here to avoid train/serve skew).
- Trained model artifacts produced by Part 03.

**Output:** `predictions.parquet` / `predictions.csv` with:
- keys: `train_id`, `station_code`, `event_time`
- `risk_delay_soon` (calibrated if calibrator exists)
- `p50_delay`, `p90_delay` (post-trigger only)
- `outlook_status` = `PRE` / `POST`


In [9]:
import pandas as pd
import numpy as np
import datetime as dt
import os

# -----------------------------
# Configuration
# -----------------------------
CANONICAL_PATH = os.getenv("CANONICAL_PATH", "data/train_stop_events_labeled.parquet")

# Optional: restrict inference to a window (UTC naive timestamps).
INFER_START = os.getenv("INFER_START", "")  
INFER_END   = os.getenv("INFER_END", "")    
DEFAULT_LOOKBACK_DAYS = int(os.getenv("DEFAULT_LOOKBACK_DAYS", "7"))

HORIZON_MIN = int(os.getenv("HORIZON_MIN", "60"))
DELAY_THRESHOLD_MIN = int(os.getenv("DELAY_THRESHOLD_MIN", "10"))
ROLL_WINDOWS_MIN = [30, 60]
WEATHER_ROLL_WINDOWS_H = [3, 6] 

# -----------------------------
# 1. Load Data
# -----------------------------
if os.path.exists(CANONICAL_PATH):
    print(f"Loading data from {CANONICAL_PATH}...")
    df = pd.read_parquet(CANONICAL_PATH)
else:
    raise FileNotFoundError(f"Could not find input file at {CANONICAL_PATH}.")

# 2. Convert timestamps
if "event_time" in df.columns:
    df["event_time"] = pd.to_datetime(df["event_time"])

# 3. Filter Window (Batch Inference Logic)
if INFER_START and INFER_END:
    print(f"Filtering for window: {INFER_START} to {INFER_END}")
    mask = (df["event_time"] >= pd.to_datetime(INFER_START)) & (df["event_time"] < pd.to_datetime(INFER_END))
    df = df.loc[mask].copy()
else:
    print(f"No explicit window set. Selecting last {DEFAULT_LOOKBACK_DAYS} days.")
    max_time = df["event_time"].max()
    min_time = max_time - pd.Timedelta(days=DEFAULT_LOOKBACK_DAYS)
    df = df.loc[df["event_time"] >= min_time].copy()

print(f"✅ Data ready for inference. Rows: {len(df)}")

# -----------------------------
# 4. FIX: Rename & Print Weather
# -----------------------------
print("\n🔍 Checking for weather columns...")

# Map raw names to what features.py expects
weather_rename_map = {
    "temperature_2m": "weather_temperature_2m",
    "precipitation": "weather_precipitation",
    "rain": "weather_rain",
    "snowfall": "weather_snowfall",
    "windspeed_10m": "weather_windspeed_10m"
}

# Apply renaming
renamed_count = 0
for old_name, new_name in weather_rename_map.items():
    if old_name in df.columns:
        df.rename(columns={old_name: new_name}, inplace=True)
        renamed_count += 1

# Verify results
current_weather_cols = [c for c in df.columns if c.startswith("weather_")]

if current_weather_cols:
    print(f"✅ Success! Found and renamed {len(current_weather_cols)} weather columns.")
    print("Columns:", current_weather_cols)
    print("\nSample Weather Data:")
    display(df[current_weather_cols].head(5))
else:
    print("❌ WARNING: No weather columns found. Check if 'temperature_2m' etc. exist in the parquet file.")
    print("Available columns:", list(df.columns))

Loading data from data/train_stop_events_labeled.parquet...
No explicit window set. Selecting last 7 days.
✅ Data ready for inference. Rows: 76076

🔍 Checking for weather columns...
✅ Success! Found and renamed 6 weather columns.
Columns: ['weather_temperature_2m', 'weather_precipitation', 'weather_rain', 'weather_snowfall', 'weather_windspeed_10m', 'weather_time']

Sample Weather Data:


,weather_temperature_2m,weather_precipitation,weather_rain,weather_snowfall,weather_windspeed_10m,weather_time
0,0.3,0.0,0.0,0.0,6.9,2026-01-02
1,0.3,0.0,0.0,0.0,6.9,2026-01-02
2,0.3,0.0,0.0,0.0,6.9,2026-01-02
3,0.3,0.0,0.0,0.0,6.9,2026-01-02
4,0.3,0.0,0.0,0.0,6.9,2026-01-02


## 🧱 Feature engineering (identical to Part 02)

In [10]:
# --- Feature Engineering Phase ---
# We use the shared logic from features.py to ensure 
# Training and Inference generate the exact same columns.

print("⚡️ Engineering features using 'features.py'...")

# 1. Clean & Rename (Safety Check)
# Ensure weather columns are named correctly before starting
weather_rename_map = {
    "temperature_2m": "weather_temperature_2m",
    "precipitation": "weather_precipitation",
    "rain": "weather_rain",
    "snowfall": "weather_snowfall",
    "windspeed_10m": "weather_windspeed_10m"
}
existing_rename = {k: v for k, v in weather_rename_map.items() if k in df.columns}
if existing_rename:
    df = df.rename(columns=existing_rename)

# 2. Build Features
# This one function call does ALL the math (Calendar, Lags, Rolling, Weather)
df_feat = build_features(df)

print("✅ Feature engineering complete.")
print("Feature table shape:", df_feat.shape)
display(df_feat.head())

⚡️ Engineering features using 'features.py'...


✅ Feature engineering complete.
Feature table shape: (76076, 57)


,event_time,ActivityId,ActivityType,train_id,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,weather_temperature_2m_rollmean_3h,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h
0,2026-01-02 00:29:00,1500adde-075d-66fb-08de-3de279b4d082,Ankomst,10994,2026-01-02 00:29:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:30:00+01:00,Arnc,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-02 00:29:00,1500adde-075d-66fb-08de-3de279b4d083,Avgang,10994,2026-01-02 00:29:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:32:00+01:00,2026-01-02 00:32:00+01:00,Arnc,3.0,...,0.300000,0.0,0.0,0.0,6.900000,0.300000,0.0,0.0,0.0,6.900000
2,2026-01-02 00:44:00,1500adde-075d-66fb-08de-3de2c0729363,Avgang,2285,2026-01-02 00:44:00+01:00,2026-01-02 00:47:00+01:00,2026-01-02 00:45:00+01:00,2026-01-02 00:45:00+01:00,Arnc,1.0,...,0.300000,0.0,0.0,0.0,6.900000,0.300000,0.0,0.0,0.0,6.900000
3,2026-01-02 00:44:00,1500adde-075d-66fb-08de-3de2c0729362,Ankomst,2285,2026-01-02 00:44:00+01:00,2026-01-02 00:45:00+01:00,2026-01-02 00:44:00+01:00,2026-01-02 00:44:00+01:00,Arnc,0.0,...,0.166667,0.0,0.0,0.0,6.266667,0.166667,0.0,0.0,0.0,6.266667
4,2026-01-02 04:44:00,1500adde-075d-66fb-08de-3de2b72354bf,Ankomst,2205,2026-01-02 04:44:00+01:00,NaT,2026-01-02 04:44:00+01:00,2026-01-02 04:44:00+01:00,Arnc,0.0,...,NaN,NaN,NaN,NaN,NaN,0.100000,0.0,0.0,0.0,5.950000


In [11]:
import pandas as pd
import numpy as np
import os
from typing import Optional

def load_inference_events(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find inference dataset at: {path}")
    d = pd.read_parquet(path)
    if "event_time" not in d.columns:
        raise ValueError("Expected column 'event_time' in canonical dataset.")
    d["event_time"] = pd.to_datetime(d["event_time"], errors="coerce")
    d = d.dropna(subset=["event_time"]).copy()
    return d

def parse_ts(s: str) -> Optional[pd.Timestamp]:
    s = (s or "").strip()
    if not s:
        return None
    return pd.to_datetime(s, errors="raise")

# 1. Load Data
events = load_inference_events(CANONICAL_PATH)

# --- CRITICAL FIX: RENAME WEATHER COLUMNS ---
# This ensures features.py can find them later
weather_rename_map = {
    "temperature_2m": "weather_temperature_2m",
    "precipitation": "weather_precipitation",
    "rain": "weather_rain",
    "snowfall": "weather_snowfall",
    "windspeed_10m": "weather_windspeed_10m"
}
# Only rename if they exist (safe check)
existing_rename = {k: v for k, v in weather_rename_map.items() if k in events.columns}
if existing_rename:
    print(f"🔄 Renaming {len(existing_rename)} weather columns for consistency.")
    events = events.rename(columns=existing_rename)
# ---------------------------------------------

# 2. Filter to inference window
start_ts = parse_ts(INFER_START)
end_ts = parse_ts(INFER_END)

if start_ts is not None:
    events = events[events["event_time"] >= start_ts]
if end_ts is not None:
    events = events[events["event_time"] < end_ts]

if start_ts is None and end_ts is None:
    # default: last DEFAULT_LOOKBACK_DAYS present in data
    max_t = events["event_time"].max()
    min_t = max_t - pd.Timedelta(days=DEFAULT_LOOKBACK_DAYS)
    events = events[events["event_time"] >= min_t]
    print(f"No INFER_START/INFER_END set -> using last {DEFAULT_LOOKBACK_DAYS} days: {min_t} → {max_t}")

events = events.sort_values(["event_time", "station_code", "train_id"]).reset_index(drop=True)
print("Inference rows:", len(events))

# 3. Verify Weather Columns exist
w_cols = [c for c in events.columns if "weather_" in c]
print(f"✅ Active Weather Columns: {w_cols}")

display(events.head())

🔄 Renaming 5 weather columns for consistency.
No INFER_START/INFER_END set -> using last 7 days: 2025-12-29 16:54:00 → 2026-01-05 16:54:00
Inference rows: 76076
✅ Active Weather Columns: ['weather_temperature_2m', 'weather_precipitation', 'weather_rain', 'weather_snowfall', 'weather_windspeed_10m', 'weather_time']


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,train_run_id,final_delay_min,additional_delay_min,y_delay_within_horizon,weather_temperature_2m,weather_precipitation,weather_rain,weather_snowfall,weather_windspeed_10m,weather_time
0,1500adde-075d-66fb-08de-3de2e9c51bd7,Avgang,2983,2026-01-02 00:00:00,2026-01-02 00:00:00+01:00,NaT,2026-01-02 00:00:00+01:00,2026-01-02 00:00:00+01:00,Mr,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
1,1500adde-075d-66fb-08de-3de2c860a960,Avgang,2477,2026-01-02 00:03:00,2026-01-02 00:03:00+01:00,NaT,2026-01-02 00:03:00+01:00,2026-01-02 00:03:00+01:00,Söc,0.0,...,2477_2026-01-02,-2.0,-2.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
2,1500adde-075d-66fb-08de-3de2e9c6a56c,Avgang,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Rs,1.0,...,2983_2026-01-02,0.0,-1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
3,1500adde-075d-66fb-08de-3de2e9c6a56b,Ankomst,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:04:00+01:00,2026-01-02 00:04:00+01:00,Rs,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
4,1500adde-075d-66fb-08de-3de314d64fcf,Avgang,7879,2026-01-02 00:05:00,2026-01-02 00:05:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Arnn,0.0,...,7879_2026-01-02,1.0,1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02


## 🧮 Build model-ready features (no leakage)

In [13]:
import json

FEATURE_METADATA_PATH = "data/feature_pipeline_outputs/feature_metadata.json"
# --- 1. Load Feature Metadata (The "Contract" from Part 2) ---
# We need to know EXACTLY which columns the model expects.
if os.path.exists(FEATURE_METADATA_PATH):
    with open(FEATURE_METADATA_PATH, "r") as f:
        meta = json.load(f)
        # These are the lists of columns we saved in Part 2
        pred_cols = meta["pred_feature_columns"]
        react_cols = meta["react_feature_columns"]
    print(f"✅ Loaded feature lists from metadata.")
    print(f"   Predictive features: {len(pred_cols)}")
    print(f"   Reactive features:   {len(react_cols)}")
else:
    raise FileNotFoundError(f"Could not find metadata at {FEATURE_METADATA_PATH}. Did Part 2 run successfully?")

# --- 2. Feature Selection Functions ---

def build_pred_dataset(df_feat: pd.DataFrame) -> pd.DataFrame:
    """
    Prepares dataset for Predictive Model.
    SAFETY: Selects ONLY the columns used in training. 
    Drops IDs, Targets (delay_min), and Dates to prevent leakage.
    """
    # Ensure all expected columns exist (fill missing with 0 if necessary)
    # This handles edge cases where a feature might be missing in a small batch
    missing = [c for c in pred_cols if c not in df_feat.columns]
    if missing:
        print(f"⚠️ Warning: Missing columns filled with 0: {missing}")
        for c in missing:
            df_feat[c] = 0
            
    # Return exactly the columns the model was trained on
    return df_feat[pred_cols].copy()

def build_react_dataset(df_feat: pd.DataFrame) -> pd.DataFrame:
    """
    Prepares dataset for Reactive Model.
    Filter: Only trains that are ALREADY delayed.
    """
    mask = df_feat["delay_min"] >= DELAY_THRESHOLD_MIN
    df_filtered = df_feat.loc[mask].copy()
    
    # Ensure columns exist
    missing = [c for c in react_cols if c not in df_filtered.columns]
    if missing:
        for c in missing:
            df_filtered[c] = 0
            
    return df_filtered[react_cols].copy()

print("✅ Feature selection logic ready.")

✅ Loaded feature lists from metadata.
   Predictive features: 45
   Reactive features:   49
✅ Feature selection logic ready.


In [14]:
# Assuming 'df_feat' is your engineered dataframe from the previous step:
events = df_feat 

# Build *predictive* feature table
pred_ds = build_pred_dataset(events)

# Build *reactive* feature table (post-trigger subset)
react_ds = build_react_dataset(events)

print("pred_ds shape:", pred_ds.shape)
print("react_ds shape:", react_ds.shape)
display(pred_ds.head())


pred_ds shape: (76076, 45)
react_ds shape: (9094, 49)


,ActivityType,scheduled_time,estimated_time,actual_time,observed_time,delay_min,is_canceled,Deleted,InformationOwner,Deviation,...,weather_temperature_2m_rollmean_3h,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h
0,Ankomst,2026-01-02 00:29:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:30:00+01:00,1.0,False,False,Mälardalstrafik AB,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Avgang,2026-01-02 00:29:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:32:00+01:00,2026-01-02 00:32:00+01:00,3.0,False,False,Mälardalstrafik AB,None,...,0.300000,0.0,0.0,0.0,6.900000,0.300000,0.0,0.0,0.0,6.900000
2,Avgang,2026-01-02 00:44:00+01:00,2026-01-02 00:47:00+01:00,2026-01-02 00:45:00+01:00,2026-01-02 00:45:00+01:00,1.0,False,False,SL,Kort tåg,...,0.300000,0.0,0.0,0.0,6.900000,0.300000,0.0,0.0,0.0,6.900000
3,Ankomst,2026-01-02 00:44:00+01:00,2026-01-02 00:45:00+01:00,2026-01-02 00:44:00+01:00,2026-01-02 00:44:00+01:00,0.0,False,False,SL,None,...,0.166667,0.0,0.0,0.0,6.266667,0.166667,0.0,0.0,0.0,6.266667
4,Ankomst,2026-01-02 04:44:00+01:00,NaT,2026-01-02 04:44:00+01:00,2026-01-02 04:44:00+01:00,0.0,False,False,SL,None,...,NaN,NaN,NaN,NaN,NaN,0.100000,0.0,0.0,0.0,5.950000


## 🤖 Load models (from Part 03)

In [16]:
PRED_MODEL_PATH = "data/models/predictive_model.pkl"
CALIBRATOR_PATH = "data/models/calibrator.pkl"
REACT_MODEL_PATH = "data/models/reactive_model.pkl"
if not os.path.exists(PRED_MODEL_PATH):
    raise FileNotFoundError(
        f"Missing predictive model at {PRED_MODEL_PATH}. Run 3_train_training_pipeline first."
    )
pred_model = joblib.load(PRED_MODEL_PATH)

calibrator = None
if os.path.exists(CALIBRATOR_PATH):
    calibrator = joblib.load(CALIBRATOR_PATH)
    print("Loaded calibrator:", CALIBRATOR_PATH)
else:
    print("No calibrator found (ok). Will use raw model probabilities.")

if not os.path.exists(REACT_MODEL_PATH):
    raise FileNotFoundError(
        f"Missing reactive model at {REACT_MODEL_PATH}. Run 3_train_training_pipeline first."
    )
react_bundle = joblib.load(REACT_MODEL_PATH)

# Expect react_bundle to contain quantile models if you trained them:
# { "point": model, "q10": model, "q50": model, "q90": model } OR similar.
print("Reactive bundle keys:", list(react_bundle.keys()) if isinstance(react_bundle, dict) else type(react_bundle))


Loaded calibrator: data/models/calibrator.pkl
Reactive bundle keys: ['point', 'q10', 'q50', 'q90', 'target']


In [26]:
import numpy as np
import pandas as pd

print("Predicting...")

# ------------------------------------------------------------------
# 0) Define ID columns for output
# ------------------------------------------------------------------
id_cols = ["ActivityId"]  # you want these in the output

# ------------------------------------------------------------------
# 1) Build a base table that contains IDs (from events)
#    Then add your engineered features (pred_ds) by row alignment
# ------------------------------------------------------------------
# If pred_ds was created from events in the same order, index alignment works.
# Safer: we explicitly carry IDs from events and then attach feature columns.

base = events[id_cols].copy().reset_index(drop=True)

pred_ds_full = pred_ds.copy().reset_index(drop=True)

# Defensive: same length check
if len(base) != len(pred_ds_full):
    raise RuntimeError(
        f"Length mismatch: events has {len(base)} rows, pred_ds has {len(pred_ds_full)} rows. "
        "You need a real join key (e.g., ActivityId) carried inside pred_ds."
    )

# Attach IDs to pred_ds_full
pred_ds_full = pd.concat([base, pred_ds_full], axis=1)

# ------------------------------------------------------------------
# 2) Build X_pred with EXACT columns expected by the trained pipeline
# ------------------------------------------------------------------
model_for_cols = (
    calibrator if (calibrator is not None and hasattr(calibrator, "feature_names_in_"))
    else pred_model
)

if hasattr(model_for_cols, "feature_names_in_"):
    required_cols = list(model_for_cols.feature_names_in_)
else:
    raise RuntimeError("Cannot infer required feature columns from model.")

X_pred = pred_ds_full.reindex(columns=required_cols)

# ------------------------------------------------------------------
# 3) CRITICAL SANITIZATION STEP
# ------------------------------------------------------------------
import pandas as pd
import numpy as np

# Convert datetime-like columns to numeric timestamps (seconds since epoch)
for c in X_pred.columns:
    # If it's already datetime dtype
    if pd.api.types.is_datetime64_any_dtype(X_pred[c]):
        # convert to unix seconds
        X_pred[c] = (X_pred[c].view("int64") / 1e9).astype(float)
        X_pred[c] = X_pred[c].fillna(0.0)
        continue

    # If it is object but looks like a datetime (common for *_time)
    if X_pred[c].dtype == "object" and ("time" in c.lower()):
        parsed = pd.to_datetime(X_pred[c], errors="coerce")
        if parsed.notna().any():
            X_pred[c] = (parsed.view("int64") / 1e9).astype(float)
            X_pred[c] = X_pred[c].fillna(0.0)
            continue

    # Numeric / bool → float
    if pd.api.types.is_numeric_dtype(X_pred[c]) or pd.api.types.is_bool_dtype(X_pred[c]):
        X_pred[c] = X_pred[c].astype(float).fillna(0.0)
    else:
        # Categorical/text → string
        X_pred[c] = X_pred[c].apply(lambda x: "" if pd.isna(x) else str(x))


# ------------------------------------------------------------------
# 4) Predict calibrated risk
# ------------------------------------------------------------------
if calibrator is not None and hasattr(calibrator, "predict_proba"):
    risk = calibrator.predict_proba(X_pred)[:, 1]
else:
    risk = pred_model.predict_proba(X_pred)[:, 1]

# ------------------------------------------------------------------
# 5) Save output
# ------------------------------------------------------------------
out = pred_ds_full[id_cols].copy()
out["risk_delay_soon"] = risk

out_path_parquet = "data/predictions.parquet"
out_path_csv = "data/predictions.csv"

out.to_parquet(out_path_parquet, index=False)
out.to_csv(out_path_csv, index=False)

print("✅ Saved predictions:")
print(out_path_parquet)
print(out_path_csv)

display(out.head(20))


Predicting...
✅ Saved predictions:
data/predictions.parquet
data/predictions.csv


,ActivityId,risk_delay_soon
0,1500adde-075d-66fb-08de-3de279b4d082,0.027090
1,1500adde-075d-66fb-08de-3de279b4d083,0.027090
2,1500adde-075d-66fb-08de-3de2c0729363,0.027090
3,1500adde-075d-66fb-08de-3de2c0729362,0.027090
4,1500adde-075d-66fb-08de-3de2b72354bf,0.027090
5,1500adde-075d-66fb-08de-3de2b72354c0,0.021933
6,1500adde-075d-66fb-08de-3de2b6e01571,0.027090
7,1500adde-075d-66fb-08de-3de2b6e01570,0.027090
8,1500adde-075d-66fb-08de-3de2b75e1de7,0.027090
9,1500adde-075d-66fb-08de-3de2b75ca88a,0.027090
